In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xgboost as xgb
import shap

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
df = pd.read_csv("../data/processed/features.csv")

feature_cols = [
    "cumulative_energy_MJ", "time_since_heating_s",
    *[f"Bulk_{i}_cum" for i in range(1, 16)],
    "total_bulk_cum",
    *[f"Wire_{i}_cum" for i in range(1, 10)],
    "total_wire_cum",
    "gas_total", "prev_temp", "measurement_index",
]

train_df = df[~df["is_test"]].copy()
X_train = train_df[feature_cols]
y_train = train_df["Temperature"]

print(f"Training set: {X_train.shape}")

Retraining Final Model with the same tuned hyperparameters from notebook 03 and Computing SHAP Values using TreeExplainer

In [ ]:
best_params = {
    "n_estimators": 497,
    "max_depth": 5,
    "learning_rate": 0.013482378493636503,
    "subsample": 0.8731426665585854,
    "colsample_bytree": 0.9531552082713739,
    "min_child_weight": 4,
    "reg_alpha": 1.603708599268554,
    "reg_lambda": 0.004050202712441509,
    "random_state": 42,
    "n_jobs": -1,
}

model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train, verbose=False)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base prediction): {explainer.expected_value:.1f} °C")

Verifying SHAP additivity: base value + sum of SHAP values ≈ prediction

In [ ]:
sample_idx = 0
pred = model.predict(X_train.iloc[[sample_idx]])[0]
shap_sum = explainer.expected_value + shap_values[sample_idx].sum()
print(f"Prediction:              {pred:.2f} °C")
print(f"Base + SHAP sum:         {shap_sum:.2f} °C")
print(f"Difference:              {abs(pred - shap_sum):.6f} °C")

 ### SHAP computation verification

 The SHAP values matrix has the expected shape — 12,999 rows × 31 features,
 one attribution value per feature per training observation. The expected value
 (base prediction) is 1592.0°C, matching the global mean temperature from
 Notebook 03. This is the model's prediction in the absence of any feature
 information — the starting point from which SHAP values push each prediction
 higher or lower.

 The additivity check confirms that SHAP values are exact: the base value plus
 the sum of all feature contributions reconstructs the model prediction to within
 0.0005°C. This is a property of TreeExplainer — it computes exact Shapley values
 for tree ensembles, not approximations. Every subsequent analysis in this notebook
 rests on this guarantee: when a SHAP value says "cumulative energy pushed this
 prediction up by 15°C," that attribution is mathematically exact.

### Global feature importance

In [ ]:
shap_df = pd.DataFrame(shap_values, columns=feature_cols)

mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
print("Mean |SHAP| per feature:")
print(mean_abs_shap.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
mean_abs_shap.plot(kind="barh", ax=ax, edgecolor="black", alpha=0.7)
ax.set_xlabel("Mean |SHAP value| (°C)")
ax.set_title("Global Feature Importance — Mean Absolute SHAP")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

Top 5 features — cumulative contribution

In [ ]:
total_shap = mean_abs_shap.sum()
cumulative = mean_abs_shap.cumsum() / total_shap * 100

print(f"\nCumulative contribution of top features:")
for i, (feat, pct) in enumerate(cumulative.items()):
    print(f"  {i+1}. {feat:30s} — {pct:.1f}%")
    if pct > 95:
        break

SHAP Summary Plot

In [ ]:
shap.summary_plot(shap_values, X_train, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

feature_name_map = {
    "prev_temp": "Previous temperature (°C)",
    "cumulative_energy_MJ": "Arc heating energy (MJ)",
    "time_since_heating_s": "Time since last heating (s)",
    "total_bulk_cum": "Total bulk additions (kg)",
    "Bulk_14_cum": "Bulk addition 14 (kg)",
    "gas_total": "Stirring gas (Nm³)",
    "measurement_index": "Measurement sequence",
    "Bulk_4_cum": "Bulk addition 4 (kg)",
    "Wire_1_cum": "Wire addition 1 (kg)",
    "Bulk_12_cum": "Bulk addition 12 (kg)",
}

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

mean_abs_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_train.columns
).sort_values(ascending=False)

top_features = mean_abs_shap.head(10).index.tolist()
shap_values_top = shap_values[:, top_features]

# Rename features for display
shap_values_top.feature_names = [
    feature_name_map.get(f, f) for f in top_features
]

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#F8F8F8")
ax.set_facecolor("#F8F8F8")

shap.plots.beeswarm(
    shap_values_top,
    max_display=10,
    show=False,
    plot_size=None,
    color_bar_label="Feature value",
)

ax = plt.gca()
ax.set_facecolor("#F8F8F8")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=10)
ax.set_xlabel("SHAP value — impact on predicted temperature (°C)", fontsize=11)
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)

plt.title(
    "SHAP analysis: drivers of steel temperature in the Ladle Furnace",
    fontsize=13,
    fontweight="bold",
    pad=14,
    loc="left",
)

plt.tight_layout(pad=1.5)
plt.savefig("shap_beeswarm_linkedin.png", dpi=180, bbox_inches="tight", facecolor="#F8F8F8")
plt.show()
print("Saved to shap_beeswarm_linkedin.png")

 
 ### Global importance structure

 The importance ranking reveals that the model has learned a physically coherent
 representation of LF thermodynamics. The four dominant features map directly to
 the terms of a first-principles energy balance:

 | Rank | Feature | Mean SHAP (°C) | Cumulative % | Physical role |
 |---|---|---|---|---|
 | 1 | prev_temp | 8.49 | 47.0% | Current thermal state |
 | 2 | cumulative_energy_MJ | 2.88 | 62.9% | Energy input (arc heating) |
 | 3 | time_since_heating_s | 1.62 | 71.9% | Energy loss (radiation + conduction) |
 | 4 | total_bulk_cum | 1.31 | 79.1% | Thermal sink (cold additions) |

 These four features account for 79% of the model's total attribution. The next
 seven features bring the total to 95%, adding individual material effects, gas
 usage, and process stage. The remaining 20 features contribute effectively zero
 — these are the rare materials that L1 regularization correctly suppressed.

 **prev_temp dominance** — the beeswarm confirms a strong monotonic positive
 relationship: high previous temperature pushes predictions up, low pushes them
 down, with SHAP values spanning −50°C to +80°C. This reflects the incremental
 nature of the LF process — each thermal state is a perturbation from the prior
 one, not a reset. The cluster of near-zero SHAP values at this feature
 corresponds to NaN rows (first measurements), where the model correctly receives
 no signal and must rely on the remaining features.

 **cumulative_energy_MJ** — monotonic positive relationship, as expected: more
 energy delivered to the ladle means higher temperature. SHAP values range from
 approximately −15°C to +20°C, consistent with the wide range of total energy
 inputs observed in the EDA (11 MJ to 8624 MJ). The feature adds most value
 when `prev_temp` is missing or when substantial heating occurred since the last
 measurement.

 **time_since_heating_s** — monotonic negative relationship: longer elapsed time
 since the last heating session pushes predictions down. This is the model's
 learned cooling curve — after the arc turns off, radiation from the exposed bath
 surface (proportional to T⁴) and conduction through the refractory lining
 continuously drain energy from the melt. The average contribution of 1.62°C
 appears modest because most measurements occur within 2–4 minutes of heating,
 where losses are small. The feature matters most for extended waiting scenarios.

 **total_bulk_cum** — monotonic negative relationship: more bulk material added
 means lower predicted temperature. Cold solid alloys and slag formers absorb
 enthalpy as they heat to bath temperature and dissolve, acting as thermal sinks.
 The model correctly treats cumulative bulk mass as a cooling factor.

 **Individual materials** — Bulk 14 ranks fifth overall and ahead of all other
 individual materials, suggesting it has a thermal impact per kilogram that
 differs from the average. Without column decoding its identity cannot be
 confirmed, but its high frequency and large mean addition (observed in Notebook
 01 ) are consistent with a dominant ferroalloy such as FeSiMn or FeSi. Wire 1
 (likely CaSi, nearly universal) and Wire 2 (likely Al wire) show negative SHAP
 when present — consistent with cold wire injection cooling the bath.

 **gas_total** — shows a non-monotonic pattern in the beeswarm, consistent with
 its indirect thermal role. Gas stirring does not directly add or remove heat —
 it homogenizes the bath and correlates with treatment duration. The model
 appears to use gas as a proxy for treatment complexity rather than a direct
 thermal driver.

 **measurement_index** — low index (early in the heat) pushes predictions down,
 high index (later) pushes them up. This captures the general LF trajectory:
 temperature rises through the treatment as heating accumulates. The feature
 provides coarse process stage information complementary to the specific
 cumulative features.

### SHAP dependence plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# prev_temp
shap.dependence_plot("prev_temp", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[0, 0], show=False)
axes[0, 0].set_title("prev_temp")

# cumulative_energy_MJ
shap.dependence_plot("cumulative_energy_MJ", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[0, 1], show=False)
axes[0, 1].set_title("cumulative_energy_MJ")

# time_since_heating_s
shap.dependence_plot("time_since_heating_s", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[1, 0], show=False)
axes[1, 0].set_title("time_since_heating_s")

# total_bulk_cum
shap.dependence_plot("total_bulk_cum", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[1, 1], show=False)
axes[1, 1].set_title("total_bulk_cum")

plt.tight_layout()
plt.show()

 ### Dependence plot interpretation

 **prev_temp** — near-linear positive relationship with approximately 0.7°C
 SHAP increase per 1°C increase in previous temperature. The tight scatter
 confirms this is the cleanest signal in the dataset. The interaction with
 cumulative energy is visible: at the same previous temperature, higher energy
 input (red) pushes predictions slightly higher — the model correctly compounds
 the thermal state anchor with the energy delivered since the last reading.

 **cumulative_energy_MJ** — nonlinear with saturation. SHAP rises steeply
 from 0 to ~1000 MJ, then flattens above ~2000 MJ. This reflects the
 diminishing marginal impact of energy input at high temperatures: radiation
 losses scale with T⁴, so the hotter the bath gets, the more energy is needed
 to achieve the same temperature increment. The interaction with `prev_temp`
 reinforces this — at the same cumulative energy, an already-hot melt (red)
 receives a larger positive SHAP contribution than a cold one (blue), because
 the model knows that high energy + high prior temperature compounds to even
 higher predicted temperature.

 **time_since_heating_s** — steep negative drop in the first 200–300 seconds,
 then gradual decline. The model has learned an approximate cooling curve
 consistent with radiative heat loss: the initial loss rate is highest
 immediately after heating (when the surface temperature is at its peak),
 then decelerates as the surface cools. The interaction with `total_bulk_cum`
 reveals an additive cooling effect: at the same elapsed time, heats with
 larger bulk additions (red) show more negative SHAP values — the combined
 thermal burden of time-based losses and mass-based losses is greater than
 either alone.

 **total_bulk_cum** — predominantly negative with a subtle positive bump at
 low values (0–200 kg). The negative trend is thermodynamically direct: cold
 solid material absorbs enthalpy from the liquid steel as it heats and
 dissolves. The positive bump at low bulk reflects a confound — heats with
 minimal additions tend to be simple grades that arrived close to target, so
 low bulk mass co-occurs with high temperatures without causing them. Above
 ~500 kg the relationship is clearly negative and approximately linear, with
 each additional 500 kg reducing predicted temperature by roughly 1°C — a
 magnitude consistent with the enthalpy balance for ferroalloy dissolution
 in a ~150 tonne steel ladle.

### Single-prediction explanation

Case 1: typical mid-heat measurement with prev_temp available

In [ ]:
mid_heat_candidates = train_df[
    (train_df["prev_temp"].notna()) &
    (train_df["measurement_index"] >= 3) &
    (train_df["Temperature"].between(1590, 1610)) &
    (train_df["Temperature"] >= train_df["prev_temp"]) &
    ((train_df["Temperature"] - train_df["prev_temp"]) <= 8)
].copy()

mid_heat = mid_heat_candidates.iloc[0]
mid_heat_idx = X_train.index.get_loc(mid_heat.name)

print(f"Case 1 — Mid-heat measurement (heat {int(mid_heat['key'])}, "
      f"index {int(mid_heat['measurement_index'])})")
print(f"  Actual:    {mid_heat['Temperature']:.0f} °C")
print(f"  Predicted: {model.predict(X_train.iloc[[mid_heat_idx]])[0]:.1f} °C")
print(f"  prev_temp: {mid_heat['prev_temp']:.0f} °C  "
      f"(delta: +{mid_heat['Temperature'] - mid_heat['prev_temp']:.0f} °C)")
print(f"  cumulative_energy_MJ: {mid_heat['cumulative_energy_MJ']:.0f}")
print(f"  total_bulk_cum: {mid_heat['total_bulk_cum']:.0f} kg")

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[mid_heat_idx],
        base_values=explainer.expected_value,
        data=X_train.iloc[mid_heat_idx],
        feature_names=feature_cols,
    ),
    max_display=10,
    show=False,
)
plt.title(f"Case 1 — Mid-heat (heat {int(mid_heat['key'])})")
plt.tight_layout()
plt.show()

Case 2: first measurement — no prev_temp

In [ ]:
first_meas = train_df[
    (train_df["prev_temp"].isna()) &
    (train_df["measurement_index"] == 1)
].iloc[0]
first_meas_idx = X_train.index.get_loc(first_meas.name)

print(f"\nCase 2 — First measurement (heat {int(first_meas['key'])})")
print(f"  Actual: {first_meas['Temperature']:.0f} °C")
print(f"  Predicted: {model.predict(X_train.iloc[[first_meas_idx]])[0]:.1f} °C")
print(f"  prev_temp: NaN")
print(f"  cumulative_energy_MJ: {first_meas['cumulative_energy_MJ']:.0f} MJ")

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[first_meas_idx],
        base_values=explainer.expected_value,
        data=X_train.iloc[first_meas_idx],
        feature_names=feature_cols,
    ),
    max_display=10,
    show=False,
)
plt.title(f"Case 2 — First measurement (melt {int(first_meas['key'])})")
plt.tight_layout()
plt.show()

Case 3: measurement after heavy bulk addition (high total_bulk_cum)

In [ ]:
heavy_bulk = train_df[
    (train_df["total_bulk_cum"] > train_df["total_bulk_cum"].quantile(0.95)) &
    (train_df["prev_temp"].notna())
].iloc[0]
heavy_bulk_idx = X_train.index.get_loc(heavy_bulk.name)

print(f"\nCase 3 — Heavy bulk addition (melt {int(heavy_bulk['key'])})")
print(f"  Actual: {heavy_bulk['Temperature']:.0f} °C")
print(f"  Predicted: {model.predict(X_train.iloc[[heavy_bulk_idx]])[0]:.1f} °C")
print(f"  total_bulk_cum: {heavy_bulk['total_bulk_cum']:.0f} kg")
print(f"  prev_temp: {heavy_bulk['prev_temp']:.0f} °C")

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[heavy_bulk_idx],
        base_values=explainer.expected_value,
        data=X_train.iloc[heavy_bulk_idx],
        feature_names=feature_cols,
    ),
    max_display=10,
    show=False,
)
plt.title(f"Case 3 — Heavy bulk (melt {int(heavy_bulk['key'])})")
plt.tight_layout()
plt.show()


 ### Single-prediction interpretation

 Three cases illustrate the model's behavior across distinct operating regimes.
 Case 1 was selected to represent a stable mid-heat scenario where the model
 has full information and the thermal state is evolving normally. Cases where
 the temperature drops between consecutive readings were excluded — these often
 reflect a measurement artifact from thermocouple insertion causing localized
 mixing near the refractory wall, not a genuine bulk temperature drop, and would
 misrepresent the model's actual predictive logic.

 **Case 1 — Mid-heat (heat 2, 5th measurement, error 4.9°C):**
 A stable heat in its final treatment stage — temperature has risen 4°C from
 the previous reading (1604→1608°C), confirming a heating event occurred since
 the last measurement. The waterfall shows `prev_temp` = 1604°C delivering the
 dominant push (+8.7°C), correctly anchoring the prediction above the global
 mean. Cumulative energy (+2.5°C) adds a secondary upward correction consistent
 with 734 MJ of total arc input. The remaining features make small, physically
 correct adjustments: bulk additions (−0.5°C), elapsed time since heating
 (−0.3°C), gas (−0.3°C), and individual material effects. The prediction of
 1603°C against an actual of 1608°C is a 4.9°C error — the model has the
 direction and magnitude of the thermal state correct.

 **Case 2 — First measurement (heat 1, 1st measurement, error 5.6°C):**
 The hardest case — no previous temperature anchor. `prev_temp = NaN` pushes
 the prediction down by 11.3°C, reflecting the model's learned prior that first
 measurements arrive below the global mean (steel tapped from the primary
 furnace at ~1550–1570°C). The model compensates with bulk additions (−3.1°C
 for 510 kg total, −1.8°C for Bulk 14 specifically) and measurement index
 (−0.7°C for a first reading). The final prediction of 1576.6°C is only 5.6°C
 from the actual (1571°C) — strong performance given the absence of the most
 powerful feature. The separate Bulk 14 attribution (−1.8°C beyond the aggregate)
 confirms this material has higher enthalpy of dissolution per kilogram than the
 average, consistent with a major ferroalloy like FeSiMn.

 **Case 3 — Heavy bulk (heat 12, 2nd measurement, error 6.5°C):**
 Tests the model's response to a large thermal sink. `prev_temp` = 1606°C
 anchors the prediction above the mean (+10.3°C), but this is countered by
 Bulk 12 alone (618 kg, −2.8°C) and total bulk (1273 kg, −0.6°C). The model
 avoids double-counting — the dominant bulk effect is attributed to the
 individual material column, with `total_bulk_cum` capturing only the residual
 not explained by Bulk 12 specifically. The 6.5°C under-prediction indicates
 the model is slightly too pessimistic about heavy-addition cooling: the melt
 retained more heat than predicted, likely because arc heating between additions
 compensated more than the cumulative energy feature could resolve at this
 temporal granularity.

 ## Summary

 This notebook used SHAP to explain the XGBoost model's predictions and connect
 its learned behavior to the thermodynamics and kinetics of the Ladle Furnace
 process.

 ### What the model learned

 The model discovered a physically coherent representation of LF thermal dynamics
 without being given any physics equations:

 1. **Temperature is incremental** — the previous measurement is the dominant
    predictor (47% of total attribution), correctly reflecting that the LF
    process is a sequence of small perturbations from the current thermal state.

 2. **Arc energy heats the steel** — cumulative energy input has a positive,
    saturating effect on predicted temperature (16% of attribution), consistent
    with the diminishing marginal returns of heating at elevated temperatures
    where radiation losses scale with T⁴.

 3. **Time means cooling** — elapsed time since the last heating session has a
    negative effect that follows an approximate radiative cooling curve (9% of
    attribution), with steep initial loss rates that decelerate over time.

 4. **Additions are thermal sinks** — bulk and wire additions push predictions
    down in proportion to their cumulative mass (7% + individual material effects),
    with the model learning material-specific thermal impacts that vary from the
    average.

 5. **Some materials matter individually** — Bulk 14, Bulk 12, and Bulk 4 survive
    L1 regularization and contribute separate SHAP effects beyond the aggregate
    total, indicating they have distinct enthalpies of dissolution that the model
    has learned to distinguish.

 ### What the model cannot do

 1. **First-measurement prediction** — without a previous temperature anchor,
    MAE rises to ~18°C. Arrival temperature depends on upstream variables not
    in this dataset: tapping temperature, ladle transit time, refractory
    condition, and waiting time before LF treatment.

 2. **Incremental energy resolution** — the cumulative energy feature cannot
    distinguish between energy delivered gradually over many sessions versus a
    large burst just before the current measurement. This limits the model's
    ability to predict large temperature jumps after intense heating events.

 3. **Extreme temperature recovery** — the regression-toward-the-mean effect
    means the model underestimates temperatures above ~1640°C and overestimates
    below ~1550°C, where training data is sparse.

 ### Operational relevance

 The SHAP analysis demonstrates that the model's learned relationships are
 physically trustworthy — its predictions move in the right direction for the
 right reasons. For an operator or process engineer evaluating whether to trust
 a model prediction, this is the critical assurance: the model is not exploiting
 statistical artifacts, it is encoding the same thermal logic that governs the
 actual process.

 The clear dominance of `prev_temp` also highlights where operational data
 collection matters most: ensuring that every temperature measurement is
 accurately recorded and correctly timestamped has a larger impact on model
 performance than adding any new sensor. Data quality on the existing
 measurements is worth more than new instrumentation.